## Dataset Porto Seguro Safe Driver Prediction

In [1]:
import pandas as pd
df_train = pd.read_csv('../Cleaned-Dataset/train.csv')
df_test = pd.read_csv('../Cleaned-Dataset/test.csv')

In [2]:
train_samples = int(df_train.shape[0] * 0.8)

X_train, y_train = df_train[:train_samples].drop(columns=['target']), df_train[:train_samples]['target']
X_valid, y_valid = df_train[train_samples:].drop(columns=['target']), df_train[train_samples:]['target']

test = df_test

In [3]:
# Encoding 
# Target encode only high-cardinality column
cat_cols_target = ["ps_car_11_cat"]

cat_cols_final = [col for col in X_train.columns if col.endswith("_cat")]
cat_cols_onehot = [col for col in cat_cols_final if col not in cat_cols_target]

from category_encoders import TargetEncoder, OneHotEncoder
# --- Target Encoding ---
target_encoder = TargetEncoder(cols=cat_cols_target, smoothing=0.3)

X_train = target_encoder.fit_transform(X_train, y_train)
X_valid = target_encoder.transform(X_valid)
test = target_encoder.transform(test)

# --- One Hot Encoding ---
onehot_encoder = OneHotEncoder(cols=cat_cols_onehot, use_cat_names=True)

X_train = onehot_encoder.fit_transform(X_train)
X_valid = onehot_encoder.transform(X_valid)
test = onehot_encoder.transform(test)

- optimal max_depth = 6, min_samples_split = 50 for decision Tree

## Model-1: Random Forest From Scratch with Sklearn Decision Tree


In [11]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import mode
from collections import defaultdict

class RandomForestClassifierScratchWithDecisionTreeSklearn:
    def __init__(
        self,
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        max_features='sqrt',
        bootstrap=True,
        oob_score=False,
        random_state=None,
        n_jobs=1,
        criterion='gini',
        **tree_kwargs
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features          
        self.bootstrap = bootstrap
        self.oob_score = oob_score
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.criterion = criterion
        self.tree_kwargs = tree_kwargs
        self.trees = []
        self.oob_score_ = None
        self.feature_importances_ = None

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        n_samples = X.shape[0]
        rng = np.random.RandomState(self.random_state)

        self.trees = []
        oob_predictions = [[] for _ in range(n_samples)]   # list of predictions per sample

        for i in range(self.n_estimators):
            # ====================== BOOTSTRAP SAMPLING ====================== #
            if self.bootstrap:
                sample_idx = rng.choice(n_samples, size=n_samples, replace=True)
                # OOB mask (samples never selected)
                in_bag = np.zeros(n_samples, dtype=bool)
                in_bag[sample_idx] = True
                oob_idx = np.where(~in_bag)[0]
            else:
                sample_idx = np.arange(n_samples)
                oob_idx = np.array([], dtype=int)

            # ====================== BUILD TREE WITH ALL PARAMETERS ====================== #
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features,
                criterion=self.criterion,
                random_state=rng.randint(0, 2**31 - 1) if self.random_state is not None else None,
                **self.tree_kwargs
            )
            tree.fit(X[sample_idx], y[sample_idx])
            self.trees.append(tree)

            # ====================== OOB PREDICTIONS =================================== #
            if self.oob_score and len(oob_idx) > 0:
                oob_pred = tree.predict(X[oob_idx])
                for j, idx in enumerate(oob_idx):
                    oob_predictions[idx].append(oob_pred[j])
        
        # ================== Compute OOB Score ========================================== #
        if self.oob_score:
            oob_final_pred = np.full(n_samples, -1, dtype=int)
            for i in range(n_samples):
                if len(oob_predictions[i]) > 0:
                    oob_final_pred[i] = mode(oob_predictions[i], keepdims=False).mode
            valid = oob_final_pred != -1
            self.oob_score_ = np.mean(oob_final_pred[valid] == y[valid])
        
        # =================== Feature Importances (aggregated) ======================== #
        importances = np.array([tree.feature_importances_ for tree in self.trees])
        self.feature_importances_ = np.mean(importances, axis=0)
        return self
    
    def predict(self, X):
        X = np.array(X)
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        # Majority Vote
        return mode(tree_preds, axis=0, keepdims=False).mode
    
    def predict_proba(self, X):
        X = np.array(X)
        probs = np.array([tree.predict_proba(X) for tree in self.trees])
        return np.mean(probs, axis=0)
    

In [12]:

rf_model_1 = RandomForestClassifierScratchWithDecisionTreeSklearn(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    max_features='sqrt',
    bootstrap=True,
    oob_score=True,
    random_state=42
)

rf_model_1.fit(X_train, y_train)
print("Model 1 OOB Score: ", rf_model_1.oob_score_)


Model 1 OOB Score:  0.9635843576545302


In [13]:
from sklearn.metrics import accuracy_score

y_preds_model1 = rf_model_1.predict(X_valid)
print("X_valid accuracy: ", accuracy_score(y_valid, y_preds_model1))

X_valid accuracy:  0.9634249808892585


In [14]:
# --- Test Prediction ---

test_preds_model1 = rf_model_1.predict_proba(test)[:, 1]



In [15]:
# --- create submission ---

submission = pd.DataFrame({
    "id": test['id'],
    "target": test_preds_model1.round(4)
})

In [16]:
submission.to_csv("../Final-Submission/RF-submission_model1.csv", index=False)

## Sklearn Model

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np

rf_sklearn = RandomForestClassifier(
    n_estimators=200,           # more trees = smoother
    max_depth=None,             # let trees grow full
    min_samples_split=2,
    max_features='sqrt',        # critical for decorrelation on redundant features
    bootstrap=True,
    oob_score=True,             # we will compare with our OOB
    n_jobs=-1,
    random_state=42,
    criterion='gini',
    class_weight='balanced'     # handles mild imbalance
)


In [18]:
rf_sklearn.fit(X_train, y_train)
print("Sklearn OOB score:", rf_sklearn.oob_score_)

Sklearn OOB score: 0.963582257559816


In [19]:
y_preds_sklearn = rf_sklearn.predict(X_valid)
print("valid accuracy:", accuracy_score(y_valid, y_preds_sklearn))
print("Top 5 features (should down-weight redundant & noise):", 
      np.argsort(rf_sklearn.feature_importances_)[-5:])

valid accuracy: 0.9634333812151912
Top 5 features (should down-weight redundant & noise): [83 87 30  0 86]


## Model-2: Model From Scratch 

### Decision Tree From Scratch

In [20]:
import numpy as np
from scipy.stats import mode


class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, class_label=None, proba=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.class_label = class_label
        self.proba = proba


class DecisionTreeFromScratch_Opt_Time:

    def __init__(self, max_depth=10, min_samples_leaf=10, min_samples_split=20, max_features=None, pos_weight=5):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.pos_weight = pos_weight
        self.root = None

    # ----------------------------
    # Gini
    # ----------------------------
    def _gini_from_counts(self, pos, neg):

        total = pos + neg
        if total == 0:
            return 0

        p_pos = pos / total
        p_neg = neg / total

        return 1 - (p_pos**2 + p_neg**2)

    # ----------------------------
    # Best Split
    # ----------------------------
    def _best_split(self, X, y, indices):
        m = len(indices)

        if m < self.min_samples_split:
            return None, None

        best_gini = float('inf')
        best_feature = None
        best_threshold = None

        n_features = X.shape[1]

        features = np.arange(n_features)

        if isinstance(self.max_features, int):
            features = np.random.choice(n_features, self.max_features, replace=False)

        for feature in features:
            sorted_idx = indices[np.argsort(X[indices, feature])]
            y_sorted = y[sorted_idx]
            X_sorted = X[sorted_idx, feature]

            total_pos = np.sum(y_sorted == 1)
            total_neg = m - total_pos

            left_pos = 0
            left_neg = 0

            for i in range(1, m):

                if y_sorted[i-1] == 1:
                    left_pos += 1
                else:
                    left_neg += 1

                right_pos = total_pos - left_pos
                right_neg = total_neg - left_neg

                if i < self.min_samples_leaf or (m-i) < self.min_samples_leaf:
                    continue

                if X_sorted[i] == X_sorted[i-1]:
                    continue

                gini_left = self._gini_from_counts(left_pos, left_neg)
                gini_right = self._gini_from_counts(right_pos, right_neg)

                weighted_gini = (i/m) * gini_left + ((m-i)/m) * gini_right

                if weighted_gini < best_gini:
                    best_gini = weighted_gini
                    best_feature = feature
                    best_threshold = (X_sorted[i] + X_sorted[i-1]) / 2

        return best_feature, best_threshold

    # ----------------------------
    # Build Tree
    # ----------------------------
    def _build_tree(self, X, y, indices, depth=0):

        m = len(indices)

        y_subset = y[indices]

        pos = np.sum(y_subset == 1)
        neg = m - pos

        proba = pos / (pos + neg)

        if (
            (self.max_depth is not None and depth >= self.max_depth)
            or m < self.min_samples_split
            or len(np.unique(y_subset)) == 1
        ):
            return Node(class_label=1 if proba >= (1/self.pos_weight) else 0, proba=proba)

        feature, threshold = self._best_split(X, y, indices)

        if feature is None:
            return Node(class_label=1 if proba >= (1/self.pos_weight) else 0, proba=proba)

        left_indices = indices[X[indices, feature] <= threshold]
        right_indices = indices[X[indices, feature] > threshold]

        left = self._build_tree(X, y, left_indices, depth+1)
        right = self._build_tree(X, y, right_indices, depth+1)

        return Node(feature=feature, threshold=threshold, left=left, right=right)

    # ----------------------------
    # Fit
    # ----------------------------
    def fit(self, X, y):

        X = np.array(X)
        y = np.array(y).ravel()

        indices = np.arange(len(y))
        self.root = self._build_tree(X, y, indices)

        return self

    # ----------------------------
    # Predict
    # ----------------------------
    def _predict_one(self, x, node):

        if node.class_label is not None:
            return node.class_label

        if x[node.feature] <= node.threshold:
            return self._predict_one(x, node.left)
        else:
            return self._predict_one(x, node.right)

    def predict(self, X):

        X = np.array(X)

        return np.array([self._predict_one(x, self.root) for x in X])

    # ----------------------------
    # Predict Probability
    # ----------------------------
    def _predict_proba_one(self, x, node):

        if node.proba is not None:
            return node.proba

        if x[node.feature] <= node.threshold:
            return self._predict_proba_one(x, node.left)
        else:
            return self._predict_proba_one(x, node.right)

    def predict_proba(self, X):

        X = np.array(X)

        return np.array([self._predict_proba_one(x, self.root) for x in X])


### Random Forest From Scratch

In [21]:
class RandomForestClassifierScratch:

    def __init__(
        self,
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        max_features=None,
        bootstrap=True,
        oob_score=False,
        random_state=None
    ):

        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.bootstrap = bootstrap
        self.oob_score = oob_score
        self.random_state = random_state

        self.trees = []
        self.oob_score_ = None

    def fit(self, X, y):

        X = np.asarray(X)
        y = np.asarray(y)

        n_samples = X.shape[0]

        rng = np.random.RandomState(self.random_state)

        self.trees = []

        oob_predictions = [[] for _ in range(n_samples)]

        # ====================================== TRAIN DECISION TREES =========================== #
        for i in range(self.n_estimators):

            # ============================== BOOTSTRAP SAMPLING ================================= #
            if self.bootstrap:
                sample_idx = rng.choice(n_samples, n_samples, replace=True)

                in_bag = np.zeros(n_samples, dtype=bool)
                in_bag[sample_idx] = True

                oob_idx = np.where(~in_bag)[0]
            else:
                sample_idx = np.arange(n_samples)
                oob_idx = np.array([], dtype=int)

            tree = DecisionTreeFromScratch_Opt_Time(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features
            )

            tree.fit(X[sample_idx], y[sample_idx])

            self.trees.append(tree)

            # ========================= PREDICT OOB SAMPLE ==================================== #
            if self.oob_score and len(oob_idx) > 0:

                oob_pred = tree.predict(X[oob_idx])

                for j, idx in enumerate(oob_idx):
                    oob_predictions[idx].append(oob_pred[j])

        # ======================= COMPUTE OOB SCORE ==================================== #
        if self.oob_score:

            oob_final_pred = np.full(n_samples, -1)

            for i in range(n_samples):

                if len(oob_predictions[i]) > 0:

                    oob_final_pred[i] = mode(oob_predictions[i], keepdims=False).mode

            valid = oob_final_pred != -1

            self.oob_score_ = np.mean(oob_final_pred[valid] == y[valid])

        return self

    # ----------------------------
    # Predict
    # ----------------------------
    def predict(self, X):

        X = np.array(X)

        tree_preds = np.array([tree.predict(X) for tree in self.trees])

        return mode(tree_preds, axis=0, keepdims=False).mode

    # ----------------------------
    # Predict Probabilities
    # ----------------------------
    def predict_proba(self, X):

        X = np.array(X)

        probs = np.array([tree.predict_proba(X) for tree in self.trees])

        return np.mean(probs, axis=0)


### Model Train 

In [23]:
rf_model_2 = RandomForestClassifierScratch(
    n_estimators=100,
    max_depth=6,
    min_samples_split=50,
    max_features=None,
    bootstrap=True,
    oob_score=False,
    random_state=42
)

rf_model_2.fit(X_train, y_train)
print("Model 2 OOB Score: ", rf_model_2.oob_score_)

KeyboardInterrupt: 

In [ ]:
y_preds_model2 = rf_model_2.predict(X_valid)
print("Model 2 accuracy: ", accuracy_score(y_valid, y_preds_model2))

In [ ]:
test_preds_model2 = rf_model_2.predict_proba(test)


In [ ]:
submission = pd.DataFrame({
    "id": test['id'],
    "target": test_preds_model2.round(4)
})


In [ ]:
submission.to_csv("../Final-Submission/RF-submission_model2.csv")